# 01b — Train SDXL Character LoRA (ai-toolkit)

Trains an SDXL character LoRA from your captioned reference images.

**Runtime:** A100 (40 GB) recommended. Training 2000 steps with rank 16 takes ~20–40 min on A100.
T4 (16 GB) works for smaller rank / lower batch but is tight and slower.

**Prerequisites:** Run `01a_caption_refs.ipynb` first to prepare images + captions.

**Output:** `<DRIVE_BASE>/loras/<CHARACTER_NAME>_sdxl.safetensors`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'MarcD'
TRIGGER_TOKEN  = 'sks_marcd'
TRAIN_STEPS    = 2250    # 1500–3000; bump to 2500 if identity is weak
LORA_RANK      = 32      # 32 for more identity detail (needs ~16 GB VRAM)
LEARNING_RATE  = '1.0e-4'
# ─────────────────────────────────────────────────────────────────────────

DRIVE_BASE  = '/content/drive/MyDrive/ai_character_studio'
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
LORAS_DIR   = f'{DRIVE_BASE}/loras'
OUTPUT_LORA = f'{LORAS_DIR}/{CHARACTER_NAME}_sdxl.safetensors'

import os
os.makedirs(LORAS_DIR, exist_ok=True)

refs = [f for f in os.listdir(REF_DIR) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
print(f'Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'Reference images: {len(refs)}')
if len(refs) < 10:
    print('WARNING: fewer than 10 images may cause weak identity learning.')

In [ ]:
# Install ai-toolkit
import subprocess, sys

TOOLKIT_DIR = '/content/ai-toolkit'
if not os.path.exists(TOOLKIT_DIR):
    !git clone --depth 1 https://github.com/ostris/ai-toolkit.git {TOOLKIT_DIR}
else:
    !git -C {TOOLKIT_DIR} pull --ff-only

# The core conflict:
# - Colab's system diffusers has autoencoder_rae.py which needs transformers 5.x
# - ai-toolkit needs transformers 4.46.3 (5.x breaks CLIP hidden_states)
# Fix: force-install the pinned diffusers commit (doesn't have autoencoder_rae) +
#      pin transformers 4.46.3. Use --force-reinstall to actually overwrite system versions.
!pip install -q --force-reinstall --no-deps \
    "git+https://github.com/huggingface/diffusers.git@c943837899b16cbae2f619b8dd4f7bb6f07dd81a"
!pip install -q --force-reinstall "transformers==4.46.3"

# Rest of ai-toolkit deps (allow || true since some version pins conflict)
!pip install -q -r {TOOLKIT_DIR}/requirements.txt 2>&1 | tail -3 || true

# Re-pin after requirements.txt (it tries to upgrade both)
!pip install -q --force-reinstall \
    "git+https://github.com/huggingface/diffusers.git@c943837899b16cbae2f619b8dd4f7bb6f07dd81a"
!pip install -q --force-reinstall "transformers==4.46.3"

# Remaining training deps
!pip install -q oyaml lpips "lycoris-lora==1.8.3" "optimum-quanto==0.2.4" prodigyopt \
    "peft==0.18.1" sentencepiece open_clip_torch "timm==1.0.22" "controlnet_aux==0.0.10" \
    python-slugify python-dotenv omegaconf flatten_json "av==16.0.1" huggingface_hub bitsandbytes

print('Installed. Versions:')
import importlib
for pkg in ['transformers', 'diffusers']:
    m = importlib.import_module(pkg); print(f'  {pkg}: {m.__version__}')

In [ ]:
# Check GPU
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
import yaml, json

# ai-toolkit config — job type is 'extension' (not 'train'), training_folder inside process[]
config = {
    'job': 'extension',
    'config': {
        'name': f'{CHARACTER_NAME}_sdxl',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': '/content/training_output',
            'device': 'cuda:0',
            'trigger_word': TRIGGER_TOKEN,
            'network': {
                'type': 'lora',
                'linear': LORA_RANK,
                'linear_alpha': LORA_RANK // 2,
            },
            'save': {
                'dtype': 'float16',
                'save_every': 500,
                'max_step_saves_to_keep': 3,
            },
            'datasets': [{
                'folder_path': REF_DIR,
                'caption_ext': 'txt',
                'caption_dropout_rate': 0.05,
                'shuffle_tokens': False,
                'cache_latents_to_disk': True,
                'resolution': [512, 768, 1024],
            }],
            'train': {
                'batch_size': 1,
                'steps': TRAIN_STEPS,
                'gradient_accumulation_steps': 1,
                'train_unet': True,
                'train_text_encoder': False,
                'gradient_checkpointing': True,
                'noise_scheduler': 'ddpm',
                'optimizer': 'adamw8bit',
                'lr': float(LEARNING_RATE),
                'ema_config': {'use_ema': True, 'ema_decay': 0.99},
                'dtype': 'bf16',
            },
            'model': {
                'name_or_path': 'stabilityai/stable-diffusion-xl-base-1.0',
                'is_v2': False,
                'is_xl': True,
                'is_loaded_in_4bit': False,
            },
            'sample': {
                'sampler': 'euler',
                'sample_every': 250,
                'width': 1024,
                'height': 1024,
                'prompts': [
                    f'{TRIGGER_TOKEN}, portrait photo, detailed face, neutral expression, white background',
                    f'{TRIGGER_TOKEN}, full body shot, standing, casual clothing, outdoor setting',
                    f'{TRIGGER_TOKEN}, close-up face, smiling, dramatic lighting',
                ],
                'neg': 'lowres, bad anatomy, worst quality, blurry, deformed',
                'seed': 42,
                'walk_seed': True,
                'cfg_scale': 7,
                'num_steps': 30,
            },
        }],
    },
    'meta': {'name': '[name]', 'version': '1.0'},
}

CONFIG_PATH = f'/content/{CHARACTER_NAME}_train_config.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print(f'Config written: {CONFIG_PATH}')
print(f'  job: extension | process: sd_trainer | steps: {TRAIN_STEPS} | rank: {LORA_RANK}')

In [ ]:
# Run training
import time
start = time.time()
!python {TOOLKIT_DIR}/run.py {CONFIG_PATH}
elapsed = time.time() - start
print(f'\nTraining finished in {elapsed/60:.1f} minutes.')

In [ ]:
# Copy the final LoRA to Drive
import glob, shutil

# ai-toolkit saves LoRAs to training_folder/name/
candidates = glob.glob(f'/content/training_output/{CHARACTER_NAME}_sdxl/*.safetensors')
# Pick the one with highest step number (the final checkpoint)
if not candidates:
    # Try without step suffix (some versions save as <name>.safetensors)
    candidates = glob.glob(f'/content/training_output/**/*.safetensors', recursive=True)

if candidates:
    # Sort by modification time — latest is the final
    latest = max(candidates, key=os.path.getmtime)
    shutil.copy2(latest, OUTPUT_LORA)
    print(f'✅ LoRA saved to Drive: {OUTPUT_LORA}')
    size_mb = os.path.getsize(OUTPUT_LORA) / 1024**2
    print(f'   Size: {size_mb:.1f} MB')
else:
    print('ERROR: No .safetensors found in training output. Check the training log above.')

In [ ]:
# Update metadata.json on Drive
import json
meta_path = f'{CHAR_DIR}/metadata.json'
metadata = {}
if os.path.exists(meta_path):
    with open(meta_path) as f:
        metadata = json.load(f)

metadata.update({
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'lora_path': OUTPUT_LORA,
    'train_steps': TRAIN_STEPS,
    'lora_rank': LORA_RANK,
})
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json updated: {meta_path}')
print(json.dumps(metadata, indent=2))

In [ ]:
# Quick validation — generate 2 sample images using diffusers pipeline
# (This is just a fast check; full validation is done via ComfyUI in 02_test_stills.ipynb)
from diffusers import DiffusionPipeline, AutoencoderKL
import torch
from PIL import Image

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    vae=vae, torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.load_lora_weights(OUTPUT_LORA)

prompts = [
    f'{TRIGGER_TOKEN}, portrait, detailed face, dramatic lighting',
    f'{TRIGGER_TOKEN}, full body, standing, outdoor scene',
]
images = pipe(prompts, num_inference_steps=30, guidance_scale=7.0).images

sample_dir = f'{CHAR_DIR}/samples'
os.makedirs(sample_dir, exist_ok=True)
for i, img in enumerate(images):
    path = f'{sample_dir}/validation_{i:02d}.png'
    img.save(path)
    print(f'Sample saved: {path}')

# Display
from IPython.display import display
for img in images:
    display(img.resize((512, 512)))

print('\n✅ Training complete. Check the samples above vs your reference images.')
print('If face is drifting: bump TRAIN_STEPS to 2500 or LORA_RANK to 32 and retrain.')
print('Next: run 02_test_stills.ipynb for full ComfyUI stills generation.')